# Regression

Regression in machine learning is a supervised learning technique that predicts a numerical value. It is used to predict continuous numerical values based on a set of input features mapped to output labels. It allows us to understand how the change in one or more input factors can be measured and mapped to a change in the output that is affected by the former change in the input parameters -> learns the relationship b/w input and output variables.

Regression models are used for trend forecasting, risk analysis, price prediction, etc.

## Simple Linear Regression

Linear regression is a statistical model used to find the line of best fit. What this means is that it is used to model the relationship between a dependent variable ($y$) and one or more independent variables ($x$). The line of best fit refers to the process where there is actual data and the model's predicted data, and the model tries to find the line that is closest to both the actual and predicted data, thereby minimizing the area between them.

When there is one independent variable, it is simple linear regression, and when there are multiple, it is multiple linear regression.

Linear regression assumes a linear relationship between the input variable(s) $x$ and the output variable $y$.

The formula for simple linear regression is: $$y = \beta_0 + \beta_1x + \epsilon$$
Where:
- $y$ is the dependent variable (the target)
- $x$ is the independent variable (the feature)
- $\beta_0$ is the y-intercept (the value of $y$ when $x=0$)
- $\beta_1$ is the slope of the line (the change in $y$ for a unit change in $x$).
- $\epsilon$ is the error term (the residuals of the change in $y$ that cannot be explained by $x$).

In [9]:
import sklearn as sk
import numpy as np
import torch
import statsmodels as ss
import jax
import jaxlib
import plotly.express as px

In [10]:
# creating the independent variables
weight = 0.7
bias = 0.3

# creating the data
start = 0
stop = 1
step = 0.02

X = np.arange(start, stop, step)
y = weight*X + bias

In [11]:
X[:10], y[:10], len(X), len(y)

(array([0.  , 0.02, 0.04, 0.06, 0.08, 0.1 , 0.12, 0.14, 0.16, 0.18]),
 array([0.3  , 0.314, 0.328, 0.342, 0.356, 0.37 , 0.384, 0.398, 0.412,
        0.426]),
 50,
 50)

In [12]:
X.shape, y.shape

((50,), (50,))

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train[:10], y_train[:10]

len(X_train), len(X_test)

(40, 10)

In [14]:
train_size = int(len(X) * 0.8)

X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

len(X_train), len(y_train), type(X_train)

(40, 40, numpy.ndarray)

In [15]:
def plot_predictions(X_train: np.ndarray = X_train, 
                     y_train: np.ndarray = y_train, 
                     X_test: np.ndarray = X_test, 
                     y_test: np.ndarray = y_test, 
                     predictions=None):
    fig = px.scatter(x=X_train, y=y_train, labels={"x": "X", "y": "y"}, width=600, height=400)
    fig.add_scatter(x=X_test, y=y_test, mode='markers', name="Testing data")

    if predictions is not None:
        fig.add_scatter(x=X_test, y=predictions, mode='markers', name="Predictions")
    
    fig.show()

In [16]:
plot_predictions()

In [17]:
class SimpleLinearRegression:
    def __init__(self, learning_rate=0.01, n_iters=100):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []

    def fit(self, X: np.ndarray, y: np.ndarray):
        n_samples = len(X)
        
        """we make the weights and bias start off as random values and 
        later we will update them using gradient descent"""
        self.weights = np.random.rand()
        self.bias = np.random.rand()

        for epoch in range(self.n_iters):
            # fit the model using the formula and get the y_prediction
            y_pred = self.weights * X + self.bias

            # compute the loss
            loss = np.mean((y_pred - y) ** 2)
            self.losses.append(loss)

            # computing the gradients
            dw = -(2 / n_samples) * np.sum(X * (y - y_pred)) # since bias is already there when calculating y_pred
            db = -(2 / n_samples) * np.sum(y - y_pred)

            # update the weights and bias
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            if epoch % 10 == 0:
                print(f"Epoch: {epoch}, loss: {loss: .4f}, weight: {self.weights}, bias: {self.bias}")

    def predict(self, X: np.ndarray):
        # make sure to call fit() before predict()
        if self.weights is None or self.bias is None:
            raise ValueError("Weights and bias cannot be none. Run fit() first")
        y_pred = self.weights * X + self.bias
        return y_pred


In [18]:
model = SimpleLinearRegression(0.1, 100)
model.fit(X_train, y_train)

Epoch: 0, loss:  0.1788, weight: 0.11530677071136103, bias: 0.22454899348227514
Epoch: 10, loss:  0.0111, weight: 0.2594473488709661, bias: 0.45761259178645475
Epoch: 20, loss:  0.0085, weight: 0.304933961358243, bias: 0.4598927697064078
Epoch: 30, loss:  0.0071, weight: 0.3402635305800475, bias: 0.4469426847132671
Epoch: 40, loss:  0.0059, weight: 0.3720367547095582, bias: 0.43406072034304316
Epoch: 50, loss:  0.0049, weight: 0.4009752384330994, bias: 0.42223851115939515
Epoch: 60, loss:  0.0041, weight: 0.4273582463547343, bias: 0.41145387907207553
Epoch: 70, loss:  0.0034, weight: 0.4514133315468858, bias: 0.4016203781464429
Epoch: 80, loss:  0.0028, weight: 0.4733460347232217, bias: 0.39265445500423063
Epoch: 90, loss:  0.0023, weight: 0.4933436234537276, bias: 0.38447958966671486


In [19]:
y_preds = model.predict(X_test)

In [20]:
plot_predictions(predictions=y_preds)

In [25]:
model_2 = SimpleLinearRegression(0.1, 200)
model_2.fit(X_train, y_train)
model_2_y_preds = model_2.predict(X_test)
plot_predictions(predictions=model_2_y_preds)

Epoch: 0, loss:  0.0142, weight: 0.5731358771574355, bias: 0.2615821633466703
Epoch: 10, loss:  0.0005, weight: 0.6108995854355224, bias: 0.32996935867831634
Epoch: 20, loss:  0.0003, weight: 0.6206604618900451, bias: 0.3319719869961977
Epoch: 30, loss:  0.0003, weight: 0.6277963626650728, bias: 0.32948332161075333
Epoch: 40, loss:  0.0002, weight: 0.6341765696817157, bias: 0.32690577033681506
Epoch: 50, loss:  0.0002, weight: 0.6399848387167042, bias: 0.32453358295028056
Epoch: 60, loss:  0.0002, weight: 0.6452800020071371, bias: 0.32236911613398045
Epoch: 70, loss:  0.0001, weight: 0.6501079290831269, bias: 0.3203955066463269
Epoch: 80, loss:  0.0001, weight: 0.6545098867908867, bias: 0.31859601987603664
Epoch: 90, loss:  0.0001, weight: 0.6585234612959069, bias: 0.31695530052739
Epoch: 100, loss:  0.0001, weight: 0.6621829197114286, bias: 0.3154593411492024
Epoch: 110, loss:  0.0001, weight: 0.6655195055739715, bias: 0.314095369665443
Epoch: 120, loss:  0.0001, weight: 0.66856170580